In [ ]:
# Option A: Simple install
#!pip install requests pandas matplotlib

In [ ]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
print("requests version:", requests.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", plt.matplotlib.__version__)


In [ ]:
stock = "sh000001"  # Replace with the desired stock code

In [ ]:
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def get_stock_data(id, scale, data_len):
    """
    Fetch K-line data from Sina Finance
    scale: 1=1min, 5=5min, 15=15min, 30=30min, 60=1hour, 240=1day
    """
    url = 'http://quotes.sina.cn/cn/api/json_v2.php/CN_MarketDataService.getKLineData?symbol={0}&scale={1}&datalen={2}'
    try:
        response = requests.get(url.format(id, scale, data_len), timeout=10)
        response.raise_for_status()
        resp_json = response.json()

        # Check if API returned valid data
        if not resp_json:
            print("API returned empty data")
            return None

        resp_json.reverse()  # Order: oldest → newest

        bar_list = []
        for dc in resp_json:
            bar = {
                'date': pd.to_datetime(dc['day']),
                'open': float(dc['open']),
                'high': float(dc['high']),
                'low': float(dc['low']),
                'close': float(dc['close']),
                'volume': float(dc['volume'])
                # removed 'amount' since it's no longer provided
            }
            bar_list.append(bar)

        df = pd.DataFrame(bar_list)
        return df

    except Exception as e:
        print(f"Error: {e}")
        return None


def add_sma(df, periods=[5, 10, 20]):
    """Add Simple Moving Average columns"""
    df = df.copy()
    for n in periods:
        df[f'SMA_{n}'] = df['close'].rolling(window=n).mean()
    return df


def plot_candlestick(df, sma_list=[5, 10, 20], title=""):
    if df is None or df.empty:
        print("No data to plot")
        return

    df = df.reset_index(drop=True)
    x = range(len(df))  # Use index for plotting position
    candle_width = 0.6

    fig, (ax1, ax2) = plt.subplots(2, 1, gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

    # Draw candlesticks
    for i, row in df.iterrows():
        color = '#26a69a' if row['close'] >= row['open'] else '#ef5350'
        # Candle body
        body_bottom = min(row['open'], row['close'])
        body_height = abs(row['close'] - row['open'])
        ax1.add_patch(Rectangle((i - candle_width / 2, body_bottom), candle_width, body_height, color=color, alpha=0.8))
        # High-low wick
        ax1.plot([i, i], [row['low'], row['high']], color=color, linewidth=1)

    # Plot SMAs
    sma_colors = ['#ff9800', '#9c27b0', '#4caf50']
    for idx, period in enumerate(sma_list):
        col_name = f'SMA_{period}'
        if col_name in df.columns:
            ax1.plot(x, df[col_name], label=f'SMA {period}', color=sma_colors[idx % 3], linewidth=1.5)

    # Format x-axis date labels
    step = max(1, len(x) // 8)  # Show ~8 labels to avoid crowding
    ax1.set_xticks(x[::step])
    ax1.set_xticklabels(df['date'].dt.strftime('%m-%d %H:%M')[::step], rotation=30, ha='right')

    ax1.set_title(title, fontsize=14)
    ax1.set_ylabel('Price')
    ax1.legend(loc='upper left')
    ax1.grid(alpha=0.2)

    # Volume chart
    ax2.bar(x, df['volume'], color='#607d8b', alpha=0.7)
    ax2.set_ylabel('Volume')
    ax2.set_xlabel('Time')
    ax2.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()


# ------------------- RUN -------------------
# Option 1: Daily chart (smoothest, best for trend view)
df = get_stock_data(f'{stock}', scale=240, data_len=60)  # 60 trading days (~3 months)

# Option 2: 5-minute intraday chart
df = get_stock_data(f'{stock}', scale=5, data_len=100)

if df is not None and not df.empty:
    df = add_sma(df, periods=[5, 10, 20])
    plot_candlestick(df, title="sh000001 Shanghai Composite Index")
    print("\n✅ Data preview:")
    print(df[['date', 'open', 'high', 'low', 'close', 'volume', 'SMA_5', 'SMA_10', 'SMA_20']].tail(8))
else:
    print("❌ No data returned. Check stock code or scale.")